## Tahap 2 — Patching-as-Instrument di L11 H16 (sumbu kausal peta kesetiaan)

Lanjutan finding 05-06: head **L11 H16** signifikan di semua 6 tipe sebagai lokasi
tetap (RSA, korelasional). Notebook ini menguji sisi KAUSAL-nya:

> Kalau aktivasi L11 H16 milik kelompok A **dicangkok** dengan aktivasi milik
> kelompok B (prompt sama persis, cuma beda kalimat demografis), apakah prediksi
> distribusi opini model **pindah ke arah distribusi ASLI kelompok B** (bukan
> sekadar berubah)?

Skor selalu lawan `responses` survei asli (Wasserstein) — konsisten dengan seluruh
proyek: patching di sini adalah **instrumen ukur kesetiaan**, bukan alat steering
(bedanya dengan llm-opinions: mereka pakai patching buat menggeser output, kami
pakai buat mengukur apakah geseran itu BENAR secara demografis).

**Kondisi per (pasangan A→B, pertanyaan):**

| kondisi | yang dicangkok | perannya |
|---|---|---|
| `baseline` | — | prediksi asli A dan B |
| `patch_L11H16` | head bintang | hipotesis utama |
| `patch_L18H14` | runner-up lintas-tipe | pembanding |
| `patch_top3` | L11H16 + L11H19 + L18H14 | efek gabungan |
| `patch_random` | 1 head acak (tetap, di luar daftar top) | kontrol negatif |
| `self_patch` | L11 H16 milik A sendiri (subset) | sanity mekanisme (harus ~= baseline) |

Ekspektasi jujur: mencangkok 1 head dari 1024, di 1 posisi token, itu tuas kecil —
efeknya mungkin kecil. Yang penting **arah dan konsistensinya** lawan kontrol acak
(uji Wilcoxon berpasangan), bukan besar mentahnya.


## Sebelum jalan: setting Kaggle

1. **Accelerator**: GPU T4 x2. **Internet: On**.
2. **Attach dataset** `opinionqa_intersectional.csv` (sama dgn notebook 07/08/09).
3. Sesi bekas crash -> **RESTART SESSION** dulu.
4. **SETELAH SELESAI, download** dari `/kaggle/working/tahap2_patching/`:
   `patching_rows.csv` (per baris eksperimen) + `patching_summary.csv` — dua-duanya
   kecil, analisis lanjutan bisa lokal.

Estimasi: load model ~10 mnt; ~7.000 forward pass pendek ~45-75 mnt. Total ~1.5 jam.


In [ ]:
!pip install -q -U "transformers>=4.44" accelerate scipy tqdm

In [ ]:
import os, sys, gc, glob, ast
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import wasserstein_distance, wilcoxon
from tqdm.auto import tqdm

sys.last_traceback = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for d in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(d)
        print(f"GPU {d}: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
        if free / total < 0.9:
            print(f"  PERINGATAN: GPU {d} tidak kosong -> RESTART SESSION dulu!")


In [ ]:
MODEL_PATH = "mistralai/Mistral-7B-v0.1"

_candidates = glob.glob("/kaggle/input/**/opinionqa_intersectional.csv", recursive=True)
if _candidates:
    DATA_PATH = _candidates[0]
elif os.path.exists("opinionqa_intersectional.csv"):
    DATA_PATH = "opinionqa_intersectional.csv"
else:
    raise FileNotFoundError("opinionqa_intersectional.csv tidak ketemu.")
print("Data:", DATA_PATH)

RANDOM_SEED = 42
TYPES_RUN = ["AGExPOLPARTY", "RELIGxPOLPARTY", "RACExRELIG"]  # kuat / naik-terbesar / terlemah
N_PAIRS = 15        # pasangan terarah (A->B) per tipe
N_QUESTIONS = 25    # pertanyaan bersama per tipe
MAX_OPTIONS = 6     # buang pertanyaan beropsi > 6 (huruf A-F)
N_SELF_PATCH = 3    # pasangan pertama utk sanity self-patch

STAR = (11, 16)             # head bintang (finding 06)
RUNNER = (18, 14)
TOP3 = [(11, 16), (11, 19), (18, 14)]

OUT_DIR = "/kaggle/working/tahap2_patching"
os.makedirs(OUT_DIR, exist_ok=True)


## 1. Data: pilih tipe, pasangan sel, dan pertanyaan per pasangan

Pertanyaan dipilih **per pasangan** (irisan pertanyaan yang dijawab A DAN B) —
coverage antar sel itu sparse, ada tipe (RACExRELIG) yang nggak punya satu pun
pertanyaan yang dijawab semua sel. Pasangan kandidat disyaratkan >= 15 pertanyaan
bersama. `options` terakhir "Refused" dibuang (panjang `responses` = panjang
`ordinal` < panjang `options`).


In [ ]:
df = pd.read_csv(DATA_PATH)
for c in ["responses", "ordinal", "options"]:
    df[c] = df[c].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["group_key"] = df["attribute"] + " :: " + df["group"]
df["n_opt"] = df["ordinal"].apply(len)

rng = np.random.default_rng(RANDOM_SEED)
plan = {}
MIN_SHARED_Q = 15
for ty in TYPES_RUN:
    sub = df[(df["attribute"] == ty) & (df["n_opt"] <= MAX_OPTIONS)]
    cells = sorted(sub["group_key"].unique().tolist())
    q_per_cell = sub.groupby("group_key")["qkey"].apply(set).to_dict()
    # pertanyaan dipilih PER PASANGAN (irisan A & B) — coverage antar sel sparse,
    # nggak ada pertanyaan yang dijawab SEMUA sel di tipe tertentu (mis. RACExRELIG)
    all_pairs = [(a, b) for a in cells for b in cells
                 if a != b and len(q_per_cell[a] & q_per_cell[b]) >= MIN_SHARED_Q]
    pick = rng.choice(len(all_pairs), size=min(N_PAIRS, len(all_pairs)), replace=False)
    pairs = [all_pairs[k] for k in pick]
    pair_questions = {}
    for (a, b) in pairs:
        shared = sorted(q_per_cell[a] & q_per_cell[b])
        if len(shared) > N_QUESTIONS:
            shared = sorted(rng.choice(shared, size=N_QUESTIONS, replace=False).tolist())
        pair_questions[(a, b)] = shared
    cells_used = sorted({c for p in pairs for c in p})
    needed = sorted({(gk, qk) for (a, b), qs in pair_questions.items()
                     for qk in qs for gk in (a, b)})
    plan[ty] = dict(cells=cells_used, pairs=pairs, pair_questions=pair_questions, needed=needed)
    n_q = sum(len(v) for v in pair_questions.values())
    print(f"[{ty}] {len(cells_used)} sel, {len(pairs)} pasangan (kandidat {len(all_pairs)}), "
          f"{n_q} (pasangan x pertanyaan), baseline unik {len(needed)}")

# lookup metadata pertanyaan & distribusi asli
qmeta = {}
real_resp = {}
for r in df.itertuples():
    qmeta[r.qkey] = (r.question, r.options[: r.n_opt], r.ordinal)
    real_resp[(r.group_key, r.qkey)] = np.array(r.responses, dtype=np.float64)


## 2. Prompt & posisi baca

Kalimat demografis pakai **template T0** (jembatan ke peta notebook 09 — L11 H16
diverifikasi kuat di T0-T3). Prompt diakhiri `Answer:` -> prediksi = distribusi
probabilitas huruf opsi di token berikutnya (readout ala SubPOP/llm-opinions).
Patch juga dipasang di posisi token terakhir itu.


In [ ]:
ATTR_LABELS = {
    "RACExRELIG":       ("race", "religion"),
    "RACExPOLPARTY":    ("race", "political party affiliation"),
    "RACExPOLIDEOLOGY": ("race", "political ideology"),
    "RELIGxPOLPARTY":   ("religion", "political party affiliation"),
    "EDUCATIONxINCOME": ("highest level of education", "household income"),
    "AGExPOLPARTY":     ("age group", "political party affiliation"),
}
LETTERS = ["A", "B", "C", "D", "E", "F"]

def demo_sentence(gk):
    ty, grp = gk.split(" :: ", 1)
    v1, v2 = grp.split(" | ", 1)
    l1, l2 = ATTR_LABELS[ty]
    return f"This survey respondent's {l1} is {v1} and their {l2} is {v2}."

def build_prompt(gk, qk):
    question, options, _ = qmeta[qk]
    lines = [demo_sentence(gk), "", f"Question: {question}"]
    for i, opt in enumerate(options):
        lines.append(f"{LETTERS[i]}) {opt}")
    lines.append("Answer:")
    return "\n".join(lines)

ty0 = TYPES_RUN[0]
gk0, qk0 = plan[ty0]["needed"][0]
print(build_prompt(gk0, qk0))


## 3. Load model, mesin patch, dan readout distribusi

In [ ]:
print(f"Loading {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="balanced", low_cpu_mem_usage=True
)
model.eval()
NUM_LAYERS = model.config.num_hidden_layers
NUM_HEADS = model.config.num_attention_heads
HEAD_DIM = model.config.hidden_size // NUM_HEADS

LETTER_IDS = [tokenizer.encode(f" {L}", add_special_tokens=False)[-1] for L in LETTERS]
assert len(set(LETTER_IDS)) == len(LETTER_IDS), "token huruf tidak unik!"
print("Letter token ids:", LETTER_IDS)

# head kontrol acak yang TETAP (di luar daftar top finding 05/06)
_forbidden = set(TOP3) | {(8, 21), (23, 13), (14, 1), (20, 14), (16, 2), (16, 10), (12, 28)}
rng_ctrl = np.random.default_rng(RANDOM_SEED + 7)
while True:
    RAND_HEAD = (int(rng_ctrl.integers(0, NUM_LAYERS)), int(rng_ctrl.integers(0, NUM_HEADS)))
    if RAND_HEAD not in _forbidden:
        break
print("Head kontrol acak:", RAND_HEAD)

CAPTURE_LAYERS = sorted({l for l, _ in TOP3} | {RAND_HEAD[0]})

# ---- mesin hook: capture donor + apply patch di token terakhir ----
_donor_capture = {}          # layer -> tensor [4096] (o_proj input token terakhir)
_active_patch = {}           # (layer) -> list of (head, vec[128])

def _oproj_prehook(layer_idx):
    def fn(module, args):
        x = args[0]
        _donor_capture[layer_idx] = x[0, -1, :].detach().float().cpu()
        patches = _active_patch.get(layer_idx)
        if patches:
            x = x.clone()
            for h, vec in patches:
                x[0, -1, h * HEAD_DIM:(h + 1) * HEAD_DIM] = vec.to(x.device, x.dtype)
            return (x,) + tuple(args[1:])
        return None
    return fn

handles = [model.model.layers[L].self_attn.o_proj.register_forward_pre_hook(_oproj_prehook(L))
           for L in CAPTURE_LAYERS]
print(f"Hook terpasang di layer {CAPTURE_LAYERS}")

@torch.no_grad()
def forward_pred(prompt, n_opt, patch_spec=None):
    """patch_spec: list of ((layer, head), donor_vec4096) -> distribusi huruf [n_opt]."""
    _active_patch.clear()
    if patch_spec:
        for (L, H), vec in patch_spec:
            seg = vec[H * HEAD_DIM:(H + 1) * HEAD_DIM]
            _active_patch.setdefault(L, []).append((H, seg))
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    logits = model(**inputs).logits[0, -1, :]
    _active_patch.clear()
    sel = logits[LETTER_IDS[:n_opt]].float()
    return torch.softmax(sel, dim=0).cpu().numpy()


## 4. Pass 1 — baseline semua (sel x pertanyaan): prediksi + donor vector

Donor = aktivasi o_proj-input token terakhir di layer 11 & 18 (dan layer head-acak),
disimpan per (sel, pertanyaan). Dipakai ulang buat semua kondisi patch.


In [ ]:
baseline_pred = {}   # (gk, qk) -> dist
donors = {}          # (gk, qk) -> {layer: vec4096}

for ty in TYPES_RUN:
    p = plan[ty]
    for (gk, qk) in tqdm(p["needed"], desc=f"baseline {ty}"):
        if (gk, qk) in baseline_pred:
            continue
        n_opt = len(qmeta[qk][2])
        pred = forward_pred(build_prompt(gk, qk), n_opt)
        baseline_pred[(gk, qk)] = pred
        donors[(gk, qk)] = {L: _donor_capture[L].clone() for L in CAPTURE_LAYERS}
print(f"{len(baseline_pred)} baseline selesai.")


## 5. Pass 2 — kondisi patch per pasangan A->B

Prompt = milik A (kalimat demografis A + pertanyaan). Donor = milik B, dari
pertanyaan yang SAMA. Yang diukur: WD prediksi ke distribusi ASLI A dan ASLI B.


In [ ]:
def wd(pred, real, ordinal):
    return wasserstein_distance(ordinal, ordinal, u_weights=pred, v_weights=real)

CONDITIONS = {
    "patch_L11H16": [STAR],
    "patch_L18H14": [RUNNER],
    "patch_top3":   TOP3,
    "patch_random": [RAND_HEAD],
}

rows = []
for ty in TYPES_RUN:
    p = plan[ty]
    for pi, (A, B) in enumerate(tqdm(p["pairs"], desc=f"patch {ty}")):
        for qk in p["pair_questions"][(A, B)]:
            question, options, ordinal = qmeta[qk]
            n_opt = len(ordinal)
            realA, realB = real_resp[(A, qk)], real_resp[(B, qk)]
            predA, predB = baseline_pred[(A, qk)], baseline_pred[(B, qk)]
            prompt_A = build_prompt(A, qk)
            base = dict(attr_type=ty, pair=f"{A} -> {B}", qkey=qk,
                        wd_A_to_realA=wd(predA, realA, ordinal),
                        wd_A_to_realB=wd(predA, realB, ordinal),
                        wd_B_to_realB=wd(predB, realB, ordinal))
            for cond, heads in CONDITIONS.items():
                spec = [((L, H), donors[(B, qk)][L]) for (L, H) in heads]
                pp = forward_pred(prompt_A, n_opt, patch_spec=spec)
                rows.append(dict(base, condition=cond,
                                 wd_patch_to_realB=wd(pp, realB, ordinal),
                                 wd_patch_to_realA=wd(pp, realA, ordinal),
                                 wd_patch_to_predB=wd(pp, predB, ordinal)))
            if pi < N_SELF_PATCH:  # sanity: cangkok A dengan A sendiri
                spec = [(STAR, donors[(A, qk)][STAR[0]])]
                pp = forward_pred(prompt_A, n_opt, patch_spec=spec)
                rows.append(dict(base, condition="self_patch",
                                 wd_patch_to_realB=wd(pp, realB, ordinal),
                                 wd_patch_to_realA=wd(pp, realA, ordinal),
                                 wd_patch_to_predB=wd(pp, predB, ordinal)))

res = pd.DataFrame(rows)
res["shift_ke_realB"] = res["wd_A_to_realB"] - res["wd_patch_to_realB"]   # + = mendekat ke ASLI B
res["shift_dari_realA"] = res["wd_patch_to_realA"] - res["wd_A_to_realA"] # + = menjauh dari ASLI A
res.to_csv(os.path.join(OUT_DIR, "patching_rows.csv"), index=False)
print(res.shape, "-> patching_rows.csv")


## 6. Rekap + uji statistik

`shift_ke_realB` > 0 = prediksi A pindah MENDEKATI distribusi asli B. Uji Wilcoxon
berpasangan: shift kondisi utama vs shift kontrol acak, di (pasangan x pertanyaan)
yang sama.


In [ ]:
summary_rows = []
print("=" * 100)
for ty in TYPES_RUN:
    r_ty = res[res["attr_type"] == ty]
    rand = r_ty[r_ty["condition"] == "patch_random"].set_index(["pair", "qkey"])["shift_ke_realB"]
    for cond in ["patch_L11H16", "patch_L18H14", "patch_top3", "patch_random", "self_patch"]:
        s = r_ty[r_ty["condition"] == cond].set_index(["pair", "qkey"])["shift_ke_realB"]
        if len(s) == 0:
            continue
        mean_shift = float(s.mean())
        pos_rate = float((s > 0).mean())
        p_vs_rand = np.nan
        if cond not in ("patch_random", "self_patch"):
            joined = pd.concat([s, rand], axis=1, keys=["c", "r"]).dropna()
            if len(joined) > 10 and not np.allclose(joined["c"], joined["r"]):
                p_vs_rand = float(wilcoxon(joined["c"], joined["r"]).pvalue)
        summary_rows.append(dict(attr_type=ty, condition=cond, n=len(s),
                                 mean_shift_ke_realB=mean_shift, persen_positif=pos_rate,
                                 p_wilcoxon_vs_random=p_vs_rand))
        p_str = "-" if np.isnan(p_vs_rand) else f"{p_vs_rand:.4f}"
        print(f"{ty:16s} {cond:14s} n={len(s):4d} | mean shift={mean_shift:+.4f} | "
              f">0: {pos_rate:.0%} | p vs random: {p_str}")

summary = pd.DataFrame(summary_rows)
summary.to_csv(os.path.join(OUT_DIR, "patching_summary.csv"), index=False)

# konteks skala: seberapa jauh sih A dan B secara asli & prediksi
print("\nKonteks skala WD (baseline):")
for ty in TYPES_RUN:
    r_ty = res[res["attr_type"] == ty].drop_duplicates(subset=["pair", "qkey"])
    print(f"{ty:16s} WD(predA, realA)={r_ty['wd_A_to_realA'].mean():.4f}  "
          f"WD(predA, realB)={r_ty['wd_A_to_realB'].mean():.4f}  "
          f"WD(predB, realB)={r_ty['wd_B_to_realB'].mean():.4f}")


## Cara baca hasil & checklist

- **Hipotesis utama lolos** kalau: `patch_L11H16` punya mean `shift_ke_realB` > 0,
  p Wilcoxon vs `patch_random` < 0.05, dan `self_patch` ~ 0 (sanity mekanisme).
- `patch_top3` > `patch_L11H16` = sinyal identitas tersebar di beberapa head (bukan
  satu); `patch_random` ~ 0 = perubahan bukan karena "diganggu apapun pindah".
- Bandingkan besar shift dengan konteks skala: `WD(predA, realB) - WD(predB, realB)`
  = jarak maksimum yang mungkin ditempuh. Shift kecil tapi konsisten & signifikan =
  instrumen bekerja; nol di mana-mana = kesetiaan kausal tidak ada di head ini
  (temuan juga — geometri korelasional != kausal).
- Hierarki tipe harus konsisten finding 06: AGE/RELIGxPP > RACExRELIG.

**Download:** `patching_rows.csv` + `patching_summary.csv` ->
`notebooks/output/11_tahap2_patching_kaggle/`. Hasil -> finding 07.
